# House Price Prediction using Machine Learning

**Goal:** predict the median house value of a California district from features such as median income, house age and location, using regression models.

**Dataset:** California Housing dataset (built into scikit-learn, 20,640 rows, 8 features). The target `MedHouseVal` is in units of $100,000.

**Steps:** load data, explore it, clean it, split it, train Linear Regression and Random Forest, evaluate with MAE / RMSE / R², and compare the two models.

## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
os.makedirs("images", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 2. Load the dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print("Shape:", df.shape)
df.head()

| Column | Meaning |
| --- | --- |
| MedInc | Median income in the block group |
| HouseAge | Median house age |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block group population |
| AveOccup | Average number of household members |
| Latitude, Longitude | Location of the block group |
| MedHouseVal | **Target**: median house value (in $100,000s) |

## 3. Understand the data

In [ ]:
df.info()

In [ ]:
df.describe().T

## 4. Data cleaning

First check for missing values and duplicate rows.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

`AveRooms`, `AveBedrms`, `Population` and `AveOccup` contain extreme values (for example, districts with hundreds of rooms per household). These outliers can distort the Linear Regression model, so they are capped at the 1st and 99th percentiles instead of being deleted.

In [ ]:
df_clean = df.copy()
outlier_cols = ["AveRooms", "AveBedrms", "Population", "AveOccup"]

for col in outlier_cols:
    low, high = df_clean[col].quantile([0.01, 0.99])
    df_clean[col] = df_clean[col].clip(low, high)

df_clean[outlier_cols].describe().T

## 5. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df_clean["MedHouseVal"], bins=40, kde=True)
plt.title("Distribution of house prices (in $100,000s)")
plt.xlabel("MedHouseVal")
plt.tight_layout()
plt.savefig("images/price_distribution.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.savefig("images/correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df_clean.sample(3000, random_state=42), x="MedInc", y="MedHouseVal", alpha=0.4)
plt.title("Median income vs house price")
plt.tight_layout()
plt.savefig("images/income_vs_price.png", dpi=150)
plt.show()

**Observations**
- Median income (`MedInc`) shows the strongest positive correlation with house price.
- Prices are capped at 5.0 ($500,000) in this dataset, which is visible as a spike at the right end of the histogram.

## 6. Split the data

80% of the data is used for training and 20% for testing.

In [ ]:
X = df_clean.drop(columns="MedHouseVal")
y = df_clean["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)

## 7. Train the models

- **Linear Regression** is the simple baseline. Features are scaled first with `StandardScaler`.
- **Random Forest** is an ensemble of decision trees that can learn non-linear patterns. It does not need scaling.

In [ ]:
lin_reg = make_pipeline(StandardScaler(), LinearRegression())
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

lin_reg.fit(X_train, y_train)
rf_reg.fit(X_train, y_train)
print("Models trained.")

## 8. Evaluate the models

Metrics used on the test set:
- **MAE**: average absolute error (in $100,000s)
- **RMSE**: square root of the mean squared error, penalises large errors more
- **R²**: share of the variance in prices explained by the model (1.0 is perfect)

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }

results = pd.DataFrame([
    evaluate("Linear Regression", lin_reg),
    evaluate("Random Forest", rf_reg),
]).round(3)

results.to_csv("results/metrics.csv", index=False)
results

### 5-fold cross-validation (R²)

Cross-validation checks that the result does not depend on one lucky train/test split.

In [ ]:
cv_rows = []
for name, model in [("Linear Regression", lin_reg), ("Random Forest", rf_reg)]:
    scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    cv_rows.append({"Model": name, "Mean CV R2": scores.mean().round(3), "Std": scores.std().round(3)})

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv("results/cross_validation.csv", index=False)
cv_results

## 9. Visualise the predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, [("Linear Regression", lin_reg), ("Random Forest", rf_reg)]):
    pred = model.predict(X_test)
    ax.scatter(y_test, pred, alpha=0.3, s=10)
    ax.plot([0, 5], [0, 5], color="red", linestyle="--", label="Perfect prediction")
    ax.set_title(name)
    ax.set_xlabel("Actual price")
    ax.set_ylabel("Predicted price")
    ax.legend()
plt.tight_layout()
plt.savefig("images/actual_vs_predicted.png", dpi=150)
plt.show()

## 10. Feature importance (Random Forest)

In [ ]:
importance = pd.Series(rf_reg.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 4))
importance.plot(kind="barh")
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("images/feature_importance.png", dpi=150)
plt.show()

## 11. Conclusion

In [ ]:
best = results.sort_values("R2", ascending=False).iloc[0]
print(f"Best model on the test set: {best['Model']} (R2 = {best['R2']}, MAE = {best['MAE']}, RMSE = {best['RMSE']})")

- The two models were compared on the same test set using MAE, RMSE and R², and checked again with 5-fold cross-validation.
- Linear Regression is fast and easy to interpret, but it can only learn linear relationships.
- Random Forest can capture non-linear patterns and interactions between features, and its feature importance shows which inputs matter most.

**Possible improvements:** hyperparameter tuning with `GridSearchCV`, trying Gradient Boosting or XGBoost, and adding new features such as rooms per person.